# Multi-Image Fusion Model Training

Trains the transformer-based multi-image fusion model that aggregates all fundus images per patient into a single diagnosis. Produces the checkpoints consumed by the aggregation comparison and multi-image coverage analysis notebooks.

In [ ]:
import sys, os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

from model import UnifiedBackboneMulti
from multi_image_dataset import MultiImageFundusDataset
from diagnosis_train_eval import train_model_multi, validate_multi

## Configuration

In [ ]:
# ── Run configuration ─────────────────────────────────────────────────────
RUN_ID     = 2
DATASET    = "mBRSET"         # "BRSET" or "mBRSET"
MODEL_NAME = "retfound_green"

# ── Data paths (edit for your environment) ───────────────────────────────
BRSET_DATA_DIR  = r"C:\Users\preet\Documents\BRSET\data"
MBRSET_DATA_DIR = r"C:\Users\preet\Documents\mBRSET\mBRSET_image_quality\data"
BRSET_IMG_ROOT  = r"C:\Users\preet\Documents\BRSET\data\resized_fundus_photos"
MBRSET_IMG_ROOT = r"C:\Users\preet\Documents\mBRSET\mbrset-a-mobile-brazilian-retinal-dataset-1.0\images"

# Checkpoints are saved as f"{CHECKPOINT_PREFIX}_img_diagnosis_model_top{rank}_BA_{ba}.pth"
CHECKPOINT_PREFIX = f"run{RUN_ID}_{DATASET}_multi_"

## Load Data

In [ ]:
if DATASET == "mBRSET":
    train_df = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_train_full.pkl"))
    val_df   = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_val_full.pkl"))
    test_df  = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_test_full.pkl"))
    img_root   = MBRSET_IMG_ROOT
    num_images = 4

elif DATASET == "BRSET":
    train_df = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_train_524.pkl"))
    val_df   = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_val_524.pkl"))
    test_df  = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_test_524.pkl"))
    img_root   = BRSET_IMG_ROOT
    num_images = 2
    # Note: do NOT filter by laterality here — both images per patient are used

patient_col = "patient"
for df in (train_df, val_df, test_df):
    df.rename(columns={"patient_id": "patient"}, inplace=True)
    df.dropna(subset=["final_icdr"], inplace=True)

print(f"Train rows: {len(train_df)}, Val rows: {len(val_df)}, Test rows: {len(test_df)}")

## Transforms

In [ ]:
if MODEL_NAME == "retfound_green":
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
else:
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = A.Compose([
    A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(392, 392),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

## Datasets & Loaders

In [ ]:
train_ds = MultiImageFundusDataset(train_df, img_root, transform=train_tf, label_col="final_icdr", patient_col=patient_col, num_images=num_images)
val_ds   = MultiImageFundusDataset(val_df,   img_root, transform=val_tf,   label_col="final_icdr", patient_col=patient_col, num_images=num_images)
test_ds  = MultiImageFundusDataset(test_df,  img_root, transform=val_tf,   label_col="final_icdr", patient_col=patient_col, num_images=num_images)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=0)

print(f"Train patients: {len(train_ds)}, Val patients: {len(val_ds)}, Test patients: {len(test_ds)}")

## Train Fusion Model

In [ ]:
def compute_class_weights(df, label_col="final_icdr", patient_col="patient"):
    """Patient-level class weights: label=1 if any image shows disease."""
    patient_labels = df.groupby(patient_col)[label_col].apply(lambda x: int((x > 0).any())).values
    counts  = np.bincount(patient_labels)
    weights = 1.0 / counts
    return torch.tensor(weights / weights.sum(), dtype=torch.float32)

device = "cuda"
model  = UnifiedBackboneMulti(model_name=MODEL_NAME, num_images=num_images)

loss_fn   = nn.CrossEntropyLoss(weight=compute_class_weights(train_df, patient_col=patient_col).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)

best_model = train_model_multi(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=20,
    patience=5,
    str_prefix=CHECKPOINT_PREFIX,
)

## Evaluate

In [ ]:
def report(split_name, loss, metrics):
    cm = metrics["conf_matrix"]
    sensitivity = cm[1, 1] / (cm[1, 0] + cm[1, 1])
    print(f"{split_name}  loss={loss:.4f}  BA={metrics['ba']:.4f}  "
          f"accuracy={metrics['accuracy']:.4f}  AUC={metrics['roc_auc']:.4f}  "
          f"AUPRC={metrics['auprc']:.4f}  F1={metrics['f1']:.4f}  sensitivity={sensitivity:.4f}")
    print(f"Confusion matrix:\n{cm}\n")

val_loss, val_metrics, *_   = validate_multi(best_model, val_loader,  loss_fn, device)
test_loss, test_metrics, *_ = validate_multi(best_model, test_loader, loss_fn, device)

report("Val ", val_loss, val_metrics)
report("Test", test_loss, test_metrics)